In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 · Validaciones para cerrar gaps
# MAGIC
# MAGIC | # | Pregunta | Por qué importa |
# MAGIC |---|---|---|
# MAGIC | 1 | ¿Pasó la compuerta `dq_gate_silver` (expect_or_fail)? | Evidencia de los 3 tipos de expectations |
# MAGIC | 2 | ¿Los tickets con productos de precio negativo cuadran? | Si cuadran, son ajustes legítimos y no distorsionan las sumas |
# MAGIC | 3 | ¿El empleado del pago es el mismo que el de los productos? | Valida que `server_id` asigna bien cada ticket al mesero |
# MAGIC | 4 | Reconciliación: fórmula ingenua vs. regla de la propina | Datos para la gráfica del dashboard |
# MAGIC
# MAGIC Conectar a **Serverless**. Requiere que el pipeline haya corrido con `dq_gate_silver`.

# COMMAND ----------

from pyspark.sql import functions as F

S = "workspace.horoz"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Compuerta de calidad (`expect_or_fail`)
# MAGIC Si esta tabla existe y tiene datos, las tres reglas pasaron: con una sola falla el pipeline habría terminado en FAILED.

# COMMAND ----------

try:
    display(spark.table(f"{S}.dq_gate_silver"))
except Exception:
    print("dq_gate_silver todavía no existe: pega el pipeline nuevo y córrelo. Las secciones 2-4 sí funcionan.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. ¿Los tickets con precio negativo cuadran?

# COMMAND ----------

display(spark.sql(f"""
WITH neg AS (
  SELECT LocationId, BusinessDate, Id,
         COUNT(*)              AS productos_negativos,
         ROUND(SUM(SoldPrice), 2) AS monto_negativo
  FROM {S}.silver_items
  WHERE SoldPrice < 0
  GROUP BY LocationId, BusinessDate, Id
)
SELECT
  COUNT(*)                                             AS tickets_con_precio_negativo,
  SUM(n.productos_negativos)                           AS productos_negativos,
  ROUND(SUM(n.monto_negativo), 2)                      AS monto_negativo_total,
  SUM(CAST(c.pago_cuadra AS INT))                      AS tickets_que_cuadran,
  ROUND(100 * AVG(CAST(c.pago_cuadra AS INT)), 2)      AS pct_cuadran
FROM neg n
JOIN {S}.silver_checks c USING (LocationId, BusinessDate, Id)
"""))

# COMMAND ----------

# ¿Qué productos son?
display(spark.sql(f"""
SELECT ItemName, COUNT(*) AS veces, ROUND(SUM(SoldPrice), 2) AS monto,
       ROUND(AVG(SoldPrice), 2) AS precio_promedio
FROM {S}.silver_items
WHERE SoldPrice < 0
GROUP BY ItemName
ORDER BY veces DESC
LIMIT 15
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. ¿El empleado del pago es el mismo que el de los productos?

# COMMAND ----------

display(spark.sql(f"""
WITH t AS (
  SELECT
    array_sort(array_distinct(filter(transform(Payments,  p -> p.EmployeeId), e -> e IS NOT NULL))) AS emp_pago,
    array_sort(array_distinct(filter(transform(ItemsSold, i -> i.EmployeeId), e -> e IS NOT NULL))) AS emp_items
  FROM {S}.bronze_checks
),
c AS (
  SELECT *,
         size(emp_pago)  > 0 AND size(emp_items) > 0 AS comparable,
         size(emp_pago)  > 0 AND emp_pago = emp_items  AS mismo
  FROM t
)
SELECT
  COUNT(*)                                                      AS tickets,
  SUM(CAST(comparable AS INT))                                  AS comparables,
  SUM(CAST(mismo AS INT))                                       AS mismo_empleado,
  ROUND(100 * SUM(CAST(mismo AS INT)) / SUM(CAST(comparable AS INT)), 2) AS pct_mismo_empleado,
  SUM(CAST(size(emp_pago) > 1 AS INT))                          AS pagos_con_varios_empleados,
  SUM(CAST(size(emp_items) > 1 AS INT))                         AS productos_con_varios_empleados,
  SUM(CAST(size(emp_pago) = 0 AS INT))                          AS sin_empleado_en_pago
FROM c
"""))

# COMMAND ----------

# Si hay diferencias: ejemplos
display(spark.sql(f"""
WITH t AS (
  SELECT LocationId, BusinessDate, Id,
    array_sort(array_distinct(filter(transform(Payments,  p -> p.EmployeeId), e -> e IS NOT NULL))) AS emp_pago,
    array_sort(array_distinct(filter(transform(ItemsSold, i -> i.EmployeeId), e -> e IS NOT NULL))) AS emp_items
  FROM {S}.bronze_checks
)
SELECT * FROM t
WHERE size(emp_pago) > 0 AND size(emp_items) > 0 AND emp_pago <> emp_items
LIMIT 20
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Reconciliación: antes y después de entender la propina
# MAGIC - **Ingenua:** pagado = venta neta + impuestos + cargo de servicio (ignora la propina).
# MAGIC - **Gratuity positivo:** se resta Gratuity tal como viene, suponiendo que es positivo.
# MAGIC - **Regla final:** pagado − propina = venta neta + impuestos + cargo, con propina = −Gratuity.

# COMMAND ----------

display(spark.sql(f"""
WITH r AS (
  SELECT
    COUNT(*) AS total,
    SUM(CAST(ABS(pagado - (venta_neta + impuestos + cargo_servicio)) > 0.05 AS INT))           AS no_ingenua,
    SUM(CAST(ABS(pagado + propina - (venta_neta + impuestos + cargo_servicio)) > 0.05 AS INT)) AS no_gratuity_positivo,
    SUM(CAST(NOT pago_cuadra AS INT))                                                          AS no_final
  FROM {S}.silver_checks
)
SELECT '1. Ingenua (sin propina)' AS metodo, no_ingenua AS tickets_que_no_cuadran,
       ROUND(100 * no_ingenua / total, 2) AS pct_no_cuadran FROM r
UNION ALL
SELECT '2. Gratuity como positivo', no_gratuity_positivo,
       ROUND(100 * no_gratuity_positivo / total, 2) FROM r
UNION ALL
SELECT '3. Regla final (Gratuity negativo)', no_final,
       ROUND(100 * no_final / total, 2) FROM r
"""))

In [0]:
%sql
SELECT table_schema, table_name, table_type, created, created_by
FROM workspace.information_schema.tables
WHERE table_schema = 'horoz'
  AND table_name NOT IN (
    'bronze_checks','bronze_employees','bronze_items','bronze_locations','bronze_revenue_centers',
    'silver_checks','silver_items','silver_voids','silver_discounts',
    'gold_daily_sales','gold_employee_activity','gold_employee_weekly_risk','gold_checks_to_review',
    'dq_gate_silver')
ORDER BY created;